## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from hazm import stopwords_list, Normalizer, WordTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer , CountVectorizer


## DataSets

In [ ]:
quality = pd.read_csv("keifiat.csv" , usecols=["comment","product_title","product_id"])
quality.isna().sum()#The Comments in this dataframe have 117 empty values.
quality.head(1)

In [ ]:
comment = pd.read_csv("comment.csv", usecols=["comment","product_id"])
comment.isna().sum()#The Comments in this dataframe have 105 empty values.
comment.head(1)


In [ ]:
orders = pd.read_csv("orders.csv",usecols=["city_name_fa","ID_Item","Quantity_item","DateTime_CartFinalize"])
orders["DateTime_CartFinalize"] = pd.to_datetime(orders["DateTime_CartFinalize"])
orders["month"] = orders["DateTime_CartFinalize"].dt.month
orders["year"] = orders["DateTime_CartFinalize"].dt.year
sum_quantity_orders = orders.groupby(["month","year" , "city_name_fa" , "ID_Item"])["Quantity_item"].sum().reset_index()


In [ ]:
history = pd.read_csv("tarikhche kharid.csv",usecols=["product_id","created_at","selling_price"])
history["created_at"] = pd.to_datetime(history["created_at"])
history= history
history["month"] = history["created_at"].dt.month
history["year"] = history["created_at"].dt.year
last_price_history = history.sort_values(["created_at"]).groupby(["product_id","month","year"]).tail(1).reset_index(drop=True)
last_price_history

In [ ]:
product = pd.read_csv("product.csv",usecols=["id","product_title_fa"])
product.head(1)

## Comment Preprocessing, TF-IDF Keyword Extraction, and Frequency Counting

In [ ]:
#All Comments

all_comments = pd.concat([comment[["product_id", "comment"]],quality[["product_id", "comment"]],],ignore_index=True).dropna(subset=["product_id", "comment"])

all_comments["product_id"] = pd.to_numeric(all_comments["product_id"], errors="coerce")

# 5 Best City-Celling
top_5 = orders.groupby("city_name_fa")["Quantity_item"].sum().nlargest(5).index


result = []
for city in top_5:
    city_orders = orders.loc[orders["city_name_fa"].eq(city)]
    top_10 = city_orders.groupby("ID_Item")["Quantity_item"].sum().nlargest(10).rename("total_quantity").reset_index()

# Comments of Top_10_Items
All_City_Comments = []

for id_item in top_10["ID_Item"]:
    comments = all_comments.loc[all_comments["product_id"].eq(id_item), ["comment", "product_id"]]
    All_City_Comments.append(comments)

All_City_Comments = pd.concat(All_City_Comments, ignore_index=True)
All_City_Comments.dropna(subset=["comment"], inplace=True)
All_City_Comments


In [ ]:
# Text PreProcessing

normalizer = Normalizer()
tokenizer = WordTokenizer()
count = CountVectorizer()

stop_words =  {token for word in stopwords_list()
               for token in tokenizer.tokenize(normalizer.normalize(word))}
extra_stop_words = {"یه", "اگه", "داره", "هست", "میشه" ,"واقعا" , "ام" , "نمیکنم" , "شدم",  "خریدم" , "نشده" , "میکنه" , "نمیشه" , "بودم" , "اصلا" , "میخوره" , "لپ" , "دی"  ,"نداره" , "میکنم"  , "یکم" , "نمیده" , "خودم" , "یکم" , "ازون" , "هستم" , "نمیدونم" , "کیفیتش" , "ایرانه" , "چیه" , "یکیشون" , "دیگه" ,"برا" ,"کنی","کلا","کش","توی","دنبال", "حدی" , "میگیره" , "باشه" ,"نگه" , "نوت" ,"اینو","برگردد"}

stop_words.update(token for word in extra_stop_words for token in tokenizer.tokenize(normalizer.normalize(word)))


def norm(text):
    norm_text = normalizer.normalize(text)
    words = tokenizer.tokenize(norm_text)


    return " ".join(word for word in words if word not in stop_words and word.isalpha())

Comments = All_City_Comments["comment"].astype(str).apply(norm)


tfidf = TfidfVectorizer(stop_words = list(stop_words))
tfidf_fit = tfidf.fit_transform(Comments)
tfidf_words = tfidf.get_feature_names_out()
scores = np.asarray(tfidf_fit.sum(axis=0)).ravel()
top_50 = scores.argsort()[::-1][:50]
pd.DataFrame({
    "word": tfidf_words[top_50],
    "tfidf_score": scores[top_50]
})

In [ ]:
comments = All_City_Comments["comment"].dropna().astype(str).apply(norm)

tfidf = TfidfVectorizer()
tfidf_fit = tfidf.fit_transform(comments)

words = tfidf.get_feature_names_out()
scores = np.asarray(tfidf_fit.sum(axis=0)).ravel()

keep = np.array([word not in stop_words for word in words])
words = words[keep]
scores = scores[keep]
top_50 = scores.argsort()[::-1][:50]
top_50_words =pd.DataFrame({ "word": words[top_50]})
top_words = words[top_50]

count = CountVectorizer()
count_matrix = count.fit_transform(comments)

count_words = count.get_feature_names_out()
count_values = np.asarray(count_matrix.sum(axis=0)).ravel()
count_by_word = dict(zip(count_words, count_values))

top_50_words = pd.DataFrame({
    "word": top_words,
    "tfidf_score": scores[top_50],
    "count": [count_by_word.get(word, 0) for word in top_words]
})


## Aggregated Statistics by City, Product, and Month

In [ ]:
sales_with_price = sum_quantity_orders.merge(
    last_price_history,
    left_on=["ID_Item", "year", "month"],
    right_on=["product_id", "year", "month"],
    how="inner")
sales_with_price["sales_amount"] = sales_with_price["selling_price"]*sales_with_price["Quantity_item"]
monthly_sales = sales_with_price.groupby(["month","year"])["sales_amount"].sum().reset_index().sort_values(["month","year"])
sales_with_price.head(5)

## Data Visualization

##### Outlier Detection Using IQR and Mean ± Standard Deviation

In [ ]:
#Outlaier Detection Using Mean +- Standard Deviation
sales = monthly_sales["sales_amount"]
monthly_sales_mean = sales.mean()
monthly_sales_std = sales.std()
Upper_1 = monthly_sales_mean + monthly_sales_std
Lower_1 = monthly_sales_mean - monthly_sales_std

#Outlaier Detection Using IQR
Q1= sales.quantile(0.25)
Q3 = sales.quantile(0.75)

IQR = Q3-Q1

Upper_2 = (Q3 + (1.5*IQR))
Lower_2 = (Q1 - 1.5*IQR)
print(Upper_1, Upper_2)

#### Plots

In [ ]:
sns.pairplot(sales_with_price,hue="month" , vars=["selling_price","sales_amount","Quantity_item"])

In [ ]:
plt.plot(monthly_sales["month"],monthly_sales["sales_amount"],marker="o")
plt.hlines(Upper_2 , 0 , 12 , linestyle="dashed" , color="red")
plt.hlines(Lower_2 , 0 , 12 , linestyle="dashed" , color="red")
plt.show()

In [ ]:
plt.boxplot(sales_with_price["sales_amount"])
plt.violinplot(sales_with_price["sales_amount"])